# IEEE-CIS Fraud Detection walkthrough

Read this notebook before running any data or modelling steps. This is an unexecuted learning implementation. All execution switches below start as `False`. The example reference score is discussed in `docs/reference_audit.md`, not presented as a result of this project.

The three questions are whether LightGBM improves on a linear model, whether engineered features improve LightGBM, and how much fraud a limited review queue can capture.


In [ ]:
from pathlib import Path

# Works when Jupyter starts in the repository root or its notebooks folder.
ROOT = Path.cwd().resolve()
if not (ROOT / "configs" / "default.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "configs" / "default.toml").exists(), "Open Jupyter from the project root."

RUN_DATA_STEPS = False
RUN_TRAINING = False
RUN_HOLDOUT_EVALUATION = False
RUN_FINAL_SUBMISSION = False
RUN_DIR = None  # Set to a saved artifacts/run_... path to resume later.


## 1 Understand the files and target

`train_transaction.csv` contains labelled transactions. `train_identity.csv` contains extra identity/device information for a subset. Test files have hidden labels. `sample_submission.csv` fixes the required output IDs and ordering.

The target is `isFraud`. A transaction is the unit of observation. `TransactionID` is a join/alignment key, while `TransactionDT` is relative elapsed seconds, not a real calendar timestamp. A left join keeps transactions without identity information.


In [ ]:
if RUN_DATA_STEPS:
    from fraud_detection.config import load_config
    from fraud_detection.data import check_files, load_data
    from fraud_detection.split import chronological_split
    cfg = load_config(ROOT / "configs" / "default.toml")
    check_files(cfg["data_dir"], training_only=True)
    data = load_data(cfg["data_dir"])
    parts = chronological_split(data, **cfg["split"])
    print({name: len(part) for name, part in parts.items()})
else:
    print("Reading mode: no competition data has been loaded.")


## 2 Explore training data first

Look for target imbalance, missing identity information, amount skew and changes across relative days. Keep later holdout labels out of exploratory model decisions. The helper writes a training summary, missingness/uniqueness table and daily transaction/fraud summaries.

Accuracy alone is inadequate: when fraud is rare, predicting every row as legitimate can give high accuracy and catch no fraud.


In [ ]:
if RUN_DATA_STEPS:
    from fraud_detection.cli import inspect_data
    inspect_data(cfg)
    training = parts["train"]
    display(training[["isFraud", "TransactionAmt", "has_identity"]].describe())
    display(training.isna().mean().sort_values(ascending=False).head(15))


## 3 Learn the chronological split

Use the earliest 60% of unique relative days for training, the next 20% for tuning, and the remaining days for holdout. These are day shares, so row shares differ. Whole-day boundaries keep equal timestamps together.

Tuning selects tree counts and the model recipe. Holdout is assessed after selection. A single split does not establish robustness across all future periods or new customers. The same card codes may recur, and label reporting delays are not simulated.


## 4 Compare raw and engineered features

Both baselines exclude ID, target and raw transaction time. They include the raw predictors and an identity-availability indicator. The engineered LightGBM adds missingness count, log amount, relative time cycles, email comparisons and training-fitted group profiles.

For example, amount divided by the training mean for the same `card1` asks whether the transaction is large relative to that code's historical profile. `card1` is not a verified customer ID. All learned profiles are frozen for tuning and holdout; no hidden-test distribution is used. These are batch training-window statistics, not fully lagged online histories.


In [ ]:
if RUN_DATA_STEPS:
    from fraud_detection.features import FeatureBuilder
    builder = FeatureBuilder(True, cfg["features"]["group_columns"])
    x_train = builder.fit_transform(parts["train"])
    x_tune = builder.transform(parts["tune"])
    print("Training feature count:", len(builder.columns_))
    print("Dropped training constants:", builder.dropped_columns_)
    display(x_train.filter(regex="amount_to_|frequency|relative_").head())
    del x_train, x_tune


## 5 Fit only when ready

The pre-specified candidates are Logistic Regression, raw-feature LightGBM and engineered-feature LightGBM. The linear model uses numeric imputation/scaling and sparse one-hot encoding. LightGBM uses native categorical splits and numeric missing values. It early-stops on tuning AUC.

The command fits preprocessing on training only, saves models and tuning predictions, and freezes the best recipe by tuning ROC-AUC. It does not score holdout. Inspect any saved convergence warnings before trusting a completed run.


In [ ]:
if RUN_TRAINING:
    from fraud_detection.config import load_config
    from fraud_detection.cli import train
    cfg = load_config(ROOT / "configs" / "default.toml")
    RUN_DIR = train(cfg)
    print("Keep this path for later:", RUN_DIR)
else:
    print("Training is disabled.")


## 6 Evaluate ranking and review capacity

ROC-AUC asks how well scores rank fraud above legitimate transactions. Average precision focuses on precision across recall levels; its random-ranking reference is fraud prevalence. Log loss and Brier assess probability quality.

At 5% review capacity, rank the holdout scores and flag the top 5%. Precision tells you the fraud share in that queue; recall tells you how much of all fraud it contains. This is an offline batch exercise, not a live threshold policy. Amount capture is a diagnostic, not measured savings.


In [ ]:
if RUN_HOLDOUT_EVALUATION:
    assert RUN_DIR is not None, "Set RUN_DIR to the path printed by training."
    from fraud_detection.cli import evaluate
    evaluate(RUN_DIR)
    import pandas as pd
    display(pd.read_csv(Path(RUN_DIR) / "evaluation" / "holdout_metrics.csv"))
    display(pd.read_csv(Path(RUN_DIR) / "evaluation" / "review_capacity.csv"))
else:
    print("Holdout evaluation is disabled.")


## 7 Create a submission only if needed

After reporting holdout results, refit the tuning-selected recipe on all labelled transactions. Use its selected tree count and rebuild preprocessing from labelled data only. Key-align probabilities to `sample_submission.csv` using `TransactionID`.

The command saves `submission.csv` locally. It does not upload anything to Kaggle. Once holdout information informs later model changes, that holdout is no longer a fresh final test for those changes.


In [ ]:
if RUN_FINAL_SUBMISSION:
    assert RUN_DIR is not None, "Set RUN_DIR to a completed and evaluated run."
    from fraud_detection.cli import submit
    submit(RUN_DIR)
else:
    print("Final refitting and submission generation are disabled.")


## 8 Write the findings

Update `reports/results.md` only with measured outputs. Compare raw LightGBM with Logistic Regression, then compare engineered LightGBM with raw LightGBM. Report time boundaries, prevalence and review precision/recall alongside AUC.

The supplied notebook printed CV AUC around 0.9296, but used a different split and preprocessing design. Do not use it to fill this project's results table. Next extensions include forward validation windows, lagged profiles, identity-coverage subgroups and calibrated decision rules.
